In [ ]:
# looking for the minimum required parameters for emulator setup
# -*- coding: utf-8 -*-
import os
import re
import json
import yaml
import pandas as pd
from pathlib import Path
from typing import Dict, List, Set

# === INPUTS ===
MAIN_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_Total_Repo_Dataset.csv"
YML_DIR  = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Min_Parameters\5.1_Emulator_Params_By_Repo.csv"

# Where to search for referenced files in cloned repos
REPO_SEARCH_ROOTS = [
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Cloned_All",
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Cloned_Sample",
]

# === Emulator presence signals ===
EMULATOR_SIGNALS = [
    r'uses:\s*reactivecircus/android-emulator-runner',          # GH Action
    r'(?m)^\s*\S*avdmanager\b',                                 # avdmanager create avd
    r'(?m)^\s*\S*sdkmanager\b[^\n"]*system-images;android-\d+', # sdkmanager system-images
    r'(?m)^\s*\S*emulator\b[^\n]*\s(-avd|@)\S+',                # emulator -avd / @AVD
    r'(?m)^\s*\S*android-wait-for-emulator\b',
    r'(?m)^\s*\S*circle-android\s+wait-for-boot\b',
]

# === Parameter patterns ===
API_PATTERNS = [
    r'\bapi[-_ ]?level\s*:\s*([0-9]{2,3})\b',                      # api-level: 30
    r'\bapiLevel\s*:\s*([0-9]{2,3})\b',                            # apiLevel: 30
    r'system-images;android-([0-9]{2,3})\b',                       # system-images;android-30;...
]
SYSIMG_PATTERNS = [
    r'\btarget\s*:\s*(google_apis(?:_playstore)?)\b',
    r'system-images;android-\d{2,3};([a-z0-9_]+)\b',               # ...;google_apis;...
    r'\b(system[-_ ]?image(?:source)?)\b\s*:\s*(google_apis(?:_playstore)?|aosp[_-]?\w*)',
]
ABI_PATTERNS = [
    r'\b(?:abi|arch)\s*:\s*(x86_64|x86|arm64[-_]?v8a|armeabi[-_]?v7a)\b',
    r'system-images;android-\d{2,3};[a-z0-9_]+;(x86_64|x86|arm64[-_]?v8a|armeabi[-_]?v7a)\b',
]
DEVICE_NAME_PATTERNS = [
    r'\bdevice\s*:\s*([A-Za-z0-9_ \-]+)\b',                        # device: pixel_5
    r'\bprofile\s*:\s*([A-Za-z0-9_ \-]+)\b',                       # profile: pixel_5
    r'\b--device\s+"?([A-Za-z0-9_ \-]+)"?',                        # avdmanager --device "pixel_5"
    r'\b(avd[-_ ]?name)\s*:\s*([A-Za-z0-9_ \-]+)\b',               # avd name: Pixel_5
]

PARAM_PATTERNS = {
    "api_level": API_PATTERNS,
    "system_image": SYSIMG_PATTERNS,
    "abi": ABI_PATTERNS,
    "device_name": DEVICE_NAME_PATTERNS,
}

# We’ll follow these referenced file types from YAML
FOLLOWABLE_EXTS = (".sh", ".bash", ".bat", ".cmd", ".ps1", ".json", ".yml", ".yaml")

# === Helpers ===
def read_text(p: Path) -> str:
    try:
        return p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        try:
            return p.read_text(encoding="latin-1", errors="ignore")
        except Exception:
            return ""

def lower_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def has_emulator_setup(text: str) -> bool:
    return any(re.search(p, text, flags=re.I) for p in EMULATOR_SIGNALS)

# Detect variable/secret-style tokens (GitHub/GitLab/Azure styles)
VAR_TOKEN_RE = re.compile(
    r'(?:(?:\$|\$\{)\s*[A-Za-z_][A-Za-z0-9_]*\s*\}?|'           # $VAR or ${VAR}
    r'\${{\s*(?:secrets|env|vars|inputs)\.[^}]+}}|'              # ${{ secrets.KEY }}
    r'%\([A-Za-z_][A-Za-z0-9_]*\)s)'                             # %(VAR)s
)

def extract_params_from_text_with_tokens(raw: str, patterns_by_key: dict):
    """Return (values, tokens) dicts for each emulator param key."""
    out_vals: Dict[str, List[str]] = {k: [] for k in patterns_by_key.keys()}
    out_tokens: Dict[str, List[str]] = {k: [] for k in patterns_by_key.keys()}

    # explicit literals
    for key, patterns in patterns_by_key.items():
        for pat in patterns:
            for m in re.finditer(pat, raw, flags=re.I | re.M):
                # prefer the first capturing group if present
                val = m.group(1) if (m.lastindex and m.group(1)) else m.group(0)
                if not val:
                    continue
                val = val.strip().strip('"').strip("'")
                if val and val.lower() not in ("api-level", "avd name"):  # ignore key echoes
                    if val not in out_vals[key]:
                        out_vals[key].append(val)

    # variable tokens near param-related text (windowed scan)
    for key in patterns_by_key.keys():
        window_re = re.compile(rf'(?i)(?:{key}|api[-_ ]?level|system[-_ ]?image|target|abi|arch|device|profile|avd[-_ ]?name)[^\n]{{0,160}}')
        for w in window_re.finditer(raw):
            segment = raw[w.start():w.end()]
            for tm in VAR_TOKEN_RE.finditer(segment):
                tok = tm.group(0)
                if tok not in out_tokens[key]:
                    out_tokens[key].append(tok)

    return out_vals, out_tokens

def merge_values(acc: Dict[str, List[str]], new: Dict[str, List[str]]) -> None:
    for k, arr in new.items():
        for v in arr:
            if v not in acc[k]:
                acc[k].append(v)

def finalize_value_and_status(values_list: List[str], tokens_list: List[str]):
    """Return (value_str, status) according to what we actually found."""
    if values_list:
        return "; ".join(values_list), "explicit"
    if tokens_list:
        return "; ".join(tokens_list), "env_or_input"
    return "", "unspecified"

# extract file references from CI YAML lines that look like run/script/command
RUN_LINE = re.compile(r'(?mi)^\s*(?:run|script|command)\s*:\s*(.+)$')
PATH_REF  = re.compile(r'(?P<path>(?:\.{0,2}/|[A-Za-z]:\\)?[A-Za-z0-9._\-/\\]+(?:' + '|'.join([re.escape(e) for e in FOLLOWABLE_EXTS]) + r'))')

def find_file_refs_from_yaml(content: str) -> List[str]:
    refs: List[str] = []
    # single-line run/script/command
    for m in RUN_LINE.finditer(content):
        line = m.group(1)
        for ref in PATH_REF.finditer(line):
            refs.append(ref.group("path"))
    # quoted config refs (e.g., -c ./.sauce/config.yml)
    for ref in re.findall(r'["\']([^"\']+\.(?:json|ya?ml|sh|bat|cmd|ps1))["\']', content, flags=re.I):
        refs.append(ref)
    # de-dup preserve order
    out, seen = [], set()
    for r in refs:
        r_norm = r.strip().strip('"').strip("'")
        if r_norm not in seen:
            seen.add(r_norm); out.append(r_norm)
    return out

def repo_key_from_full_name(full_name: str) -> str:
    return full_name.lower().replace("/", ".")

def find_repo_files(repo_key: str, roots: List[str], rel_path: str) -> List[Path]:
    """Try to locate a referenced file inside possible repo roots."""
    cand: List[Path] = []
    rel_norm = rel_path.replace("\\", "/").lstrip("./")
    for root in roots:
        owner_repo = repo_key.split(".", 1)
        variants = []
        if len(owner_repo) == 2:
            variants.append(Path(root) / owner_repo[0] / owner_repo[1] / rel_norm)
        variants.append(Path(root) / repo_key / rel_norm)
        for p in variants:
            if p.exists() and p.is_file():
                cand.append(p)
    return cand

def collect_yaml_index(yaml_dir: Path) -> Dict[str, List[Path]]:
    """Map: repo_key -> [yaml_file_paths] by filename prefix before '__'"""
    index: Dict[str, List[Path]] = {}
    for p in yaml_dir.iterdir():
        if p.is_file() and p.suffix.lower() in (".yml", ".yaml"):
            name = p.name.lower()
            if "__" in name:
                repo_key = name.split("__", 1)[0]
                index.setdefault(repo_key, []).append(p)
    return index

def main():
    # Load main CSV
    df = pd.read_csv(MAIN_CSV)
    df = lower_cols(df)

    # Ensure required columns
    for col in ["full_name", "instru_t_ci_signal"]:
        if col not in df.columns:
            raise KeyError(f"Required column '{col}' not found in {MAIN_CSV}")

    # Normalize instru_t_ci_signal to boolean
    def to_bool(x):
        if isinstance(x, str):
            return x.strip().lower() in ("true", "yes", "1")
        return bool(x)

    df["__signal"] = df["instru_t_ci_signal"].apply(to_bool)
    df_sig = df[df["__signal"] == True].copy()

    # Build index of YAML files by repo key (prefix before "__")
    yml_dir = Path(YML_DIR)
    if not yml_dir.exists():
        raise FileNotFoundError(f"YAML directory not found: {YML_DIR}")

    yaml_index = collect_yaml_index(yml_dir)

    rows = []
    for _, rec in df_sig.iterrows():
        full_name = str(rec["full_name"]).strip()
        if not full_name:
            continue
        repo_key = repo_key_from_full_name(full_name)

        acc_vals: Dict[str, List[str]]   = {k: [] for k in PARAM_PATTERNS.keys()}
        acc_tokens: Dict[str, List[str]] = {k: [] for k in PARAM_PATTERNS.keys()}
        provenance: Set[str] = set()
        emulator_seen = False

        for yml in yaml_index.get(repo_key, []):
            ytxt = read_text(yml)
            if not ytxt:
                continue

            if has_emulator_setup(ytxt):
                emulator_seen = True
                vals, toks = extract_params_from_text_with_tokens(ytxt, PARAM_PATTERNS)
                merge_values(acc_vals, vals)
                merge_values(acc_tokens, toks)
                provenance.add(str(yml))

                # Follow referenced files from YAML to catch indirect definitions
                for ref in find_file_refs_from_yaml(ytxt):
                    refs = []
                    for base in REPO_SEARCH_ROOTS:
                        refs.extend(find_repo_files(repo_key, [base], ref))
                    for rp in refs:
                        rtxt = read_text(rp)
                        if not rtxt:
                            continue

                        if rp.suffix.lower() == ".json":
                            try:
                                jtxt = json.dumps(json.loads(rtxt))
                                vals, toks = extract_params_from_text_with_tokens(jtxt, PARAM_PATTERNS)
                                merge_values(acc_vals, vals)
                                merge_values(acc_tokens, toks)
                                provenance.add(str(rp))
                                continue
                            except Exception:
                                pass

                        if rp.suffix.lower() in (".yml", ".yaml"):
                            try:
                                ytxt2 = json.dumps(yaml.safe_load(rtxt), default=str)
                                vals, toks = extract_params_from_text_with_tokens(ytxt2, PARAM_PATTERNS)
                                merge_values(acc_vals, vals)
                                merge_values(acc_tokens, toks)
                                provenance.add(str(rp))
                                continue
                            except Exception:
                                pass

                        if rp.suffix.lower() in (".sh", ".bash", ".bat", ".cmd", ".ps1"):
                            vals, toks = extract_params_from_text_with_tokens(rtxt, PARAM_PATTERNS)
                            merge_values(acc_vals, vals)
                            merge_values(acc_tokens, toks)
                            provenance.add(str(rp))

        # Only include repos where emulator device setup was detected anywhere we looked
        if not emulator_seen:
            continue

        # Finalize values + statuses
        api_level, api_status         = finalize_value_and_status(acc_vals["api_level"],    acc_tokens["api_level"])
        system_image, sysimg_status   = finalize_value_and_status(acc_vals["system_image"], acc_tokens["system_image"])
        abi, abi_status               = finalize_value_and_status(acc_vals["abi"],          acc_tokens["abi"])
        device_name, dev_status       = finalize_value_and_status(acc_vals["device_name"],  acc_tokens["device_name"])

        rows.append({
            "full_name": full_name,
            "api_level": api_level,                 "api_level_status": api_status,
            "system_image": system_image,           "system_image_status": sysimg_status,
            "abi": abi,                             "abi_status": abi_status,
            "device_name": device_name,             "device_name_status": dev_status,
            "sources": "; ".join(sorted(provenance)) if provenance else "YAML only",
        })

    out_df = pd.DataFrame(rows).sort_values(by="full_name").reset_index(drop=True)
    Path(os.path.dirname(OUTPUT_CSV)).mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved: {OUTPUT_CSV}  (rows={len(out_df)})")
    if not out_df.empty:
        print(out_df.head(20).to_string(index=False))

if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Min_Parameters\5.1_Emulator_Params_By_Repo.csv  (rows=243)
                      full_name                    api_level api_level_status                 system_image system_image_status                          abi   abi_status                  device_name device_name_status                                                                                                                                                                                                                                                                                                      sources
                a-mabe.openhiit                       35; 34         explicit                                      unspecified                       x86_64     explicit                  pixel_6_pro           explicit                                                                                                                                       

In [8]:
# minimum Parameters for Third Party testing Lab
#it search the yaml files but if a supporting file .sh or .json is referenced should check that file too
# -*- coding: utf-8 -*-
# -*- coding: utf-8 -*-
import os
import re
import json
import yaml
import pandas as pd
from pathlib import Path
from typing import Dict, List, Set

# === Inputs/Outputs ===
MAIN_CSV   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_Total_Repo_Dataset.csv"
YAML_DIR   = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Min_Parameters\5.1_ThirdPartyLab_Params_By_Repo.csv"

# 🔎 Where to look for referenced scripts/configs inside cloned repos.
# Add root directories that contain your repo working copies (e.g., owner\repo or owner.repo folders).
REPO_SEARCH_ROOTS = [
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Cloned_All",
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Cloned_Sample",
]

# === Third-party lab detection keywords ===
THIRD_PARTY_HITS = [
    r'\bgcloud\s+firebase\s+test\s+android\s+run\b',  # Firebase Test Lab CLI
    r'\bflank\s+android\s+run\b',                    # Flank (FTL)
    r'\bbrowserstack\b',                             # BrowserStack
    r'\bsaucectl\b',                                 # Sauce Labs
    r'\bdevicefarm\b',                               # AWS Device Farm
    r'\bkobiton\b',                                  # Kobiton
    r'\bbitbar\b',                                   # BitBar
]

# === Minimum parameters to extract (regex bundles) ===
# capture group 1 should be the value
PARAM_PATTERNS = {
    # Target device identifier: Firebase 'model=Pixel2', BrowserStack "device": "Google Pixel 6", Sauce 'deviceName'
    "target_device": [
        r'\bmodel\s*[:=]\s*([A-Za-z0-9._-]+)',                           # model=Pixel2 / model: Pixel2
        r'["\']device["\']\s*:\s*["\']([^"\']+)["\']',                   # "device": "Google Pixel 6"
        r'["\']deviceName["\']\s*:\s*["\']([^"\']+)["\']',               # "deviceName": "Google Pixel 4"
    ],
    # OS / API Level: Firebase 'version=30', BrowserStack "os_version": "12.0"
    "os_version_api": [
        r'\bversion\s*[:=]\s*([0-9]{2,3})\b',                            # version=30
        r'["\']os[_-]?version["\']\s*:\s*["\']([0-9.]+)["\']',           # "os_version": "12.0"
        r'\bapi\s*level\s*[:=]\s*([0-9]{2,3})\b',                        # api level: 33
    ],
    # App binary
    "app_apk_path": [
        r'\bapp\s*[:=]\s*([^\s"\'\n]+?\.(?:apk|aab))',                   # app=..., app: ...
        r'["\']app["\']\s*:\s*["\']([^"\']+\.(?:apk|aab))["\']',         # "app": "path.apk"
    ],
    # Test binary
    "test_apk_path": [
        r'\btest\s*[:=]\s*([^\s"\'\n]+?\.(?:apk|zip))',                  # test=..., test: ...
        r'["\']test["\']\s*:\s*["\']([^"\']+\.(?:apk|zip))["\']',        # "test": "path.apk"
    ],
    # Auth / credentials
    "auth_credentials": [
        r'\bGOOGLE_APPLICATION_CREDENTIALS\s*[:=]\s*([^\s"\']+\.json)\b',
        r'\bBROWSERSTACK_USERNAME\b', r'\bBROWSERSTACK_ACCESS_KEY\b',
        r'\bSAUCE_USERNAME\b', r'\bSAUCE_ACCESS_KEY\b',
        r'\bAWS_ACCESS_KEY_ID\b', r'\bAWS_SECRET_ACCESS_KEY\b',
        r'\bKOBITON_(?:USERNAME|API_KEY)\b',
    ],
}

# We’ll follow these referenced file types from YAML
FOLLOWABLE_EXTS = (".sh", ".bash", ".bat", ".cmd", ".ps1", ".json", ".yml", ".yaml")

# --- helpers ---
def read_text(p: Path) -> str:
    try:
        return p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        try:
            return p.read_text(encoding="latin-1", errors="ignore")
        except Exception:
            return ""

def lower_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def detect_third_party(raw: str) -> bool:
    text = raw.lower()
    return any(re.search(pat, text) for pat in THIRD_PARTY_HITS)

# Detect variable/secret-style tokens (GitHub/GitLab/Azure styles)
VAR_TOKEN_RE = re.compile(
    r'(?:(?:\$|\$\{)\s*[A-Za-z_][A-Za-z0-9_]*\s*\}?|'           # $VAR or ${VAR}
    r'\${{\s*(?:secrets|env|vars|inputs)\.[^}]+}}|'              # ${{ secrets.KEY }}
    r'%\([A-Za-z_][A-Za-z0-9_]*\)s)'                             # %(VAR)s
)

def extract_params_from_text_with_tokens(raw: str, patterns_by_key: dict):
    """Return (values, tokens) dicts for each param key."""
    out_vals: Dict[str, List[str]] = {k: [] for k in patterns_by_key.keys()}
    out_tokens: Dict[str, List[str]] = {k: [] for k in patterns_by_key.keys()}

    # explicit literals
    for key, patterns in patterns_by_key.items():
        for pat in patterns:
            for m in re.finditer(pat, raw, flags=re.I | re.M):
                val = m.group(1) if (m.lastindex and m.group(1)) else m.group(0)
                if not val:
                    continue
                val = val.strip().strip('"').strip("'")
                if val and val not in out_vals[key]:
                    out_vals[key].append(val)

    # variable tokens near param-related text (small windows)
    for key in patterns_by_key.keys():
        window_re = re.compile(rf'(?i)(?:{key}|app|test|device|model|version|os[_-]?version)[^\n]{{0,160}}')
        for w in window_re.finditer(raw):
            segment = raw[w.start():w.end()]
            for tm in VAR_TOKEN_RE.finditer(segment):
                tok = tm.group(0)
                if tok not in out_tokens[key]:
                    out_tokens[key].append(tok)

    return out_vals, out_tokens

def merge_values(acc: Dict[str, List[str]], new: Dict[str, List[str]]) -> None:
    for k, arr in new.items():
        for v in arr:
            if v not in acc[k]:
                acc[k].append(v)

def finalize_value_and_status(values_list: List[str], tokens_list: List[str]):
    """Return (value_str, status) according to what we actually found."""
    if values_list:
        return "; ".join(values_list), "explicit"
    if tokens_list:
        return "; ".join(tokens_list), "env_or_input"
    return "", "unspecified"

# extract file references from CI YAML lines that look like run/script/command
RUN_LINE = re.compile(r'(?mi)^\s*(?:run|script|command)\s*:\s*(.+)$')
PATH_REF  = re.compile(r'(?P<path>(?:\.{0,2}/|[A-Za-z]:\\)?[A-Za-z0-9._\-/\\]+(?:' + '|'.join([re.escape(e) for e in FOLLOWABLE_EXTS]) + r'))')

def find_file_refs_from_yaml(content: str) -> List[str]:
    refs: List[str] = []
    # single-line run/script/command
    for m in RUN_LINE.finditer(content):
        line = m.group(1)
        for ref in PATH_REF.finditer(line):
            refs.append(ref.group("path"))
    # naive scan for quoted config refs (e.g., -c ./.sauce/config.yml)
    for ref in re.findall(r'["\']([^"\']+\.(?:json|ya?ml|sh|bat|cmd|ps1))["\']', content, flags=re.I):
        refs.append(ref)
    # de-dup preserve order
    out, seen = [], set()
    for r in refs:
        r_norm = r.strip().strip('"').strip("'")
        if r_norm not in seen:
            seen.add(r_norm); out.append(r_norm)
    return out

def repo_key_from_full_name(full_name: str) -> str:
    return full_name.lower().replace("/", ".")

def find_repo_files(repo_key: str, roots: List[str], rel_path: str) -> List[Path]:
    """Try to locate a referenced file inside possible repo roots."""
    cand: List[Path] = []
    rel_norm = rel_path.replace("\\", "/").lstrip("./")
    for root in roots:
        owner_repo = repo_key.split(".", 1)
        variants = []
        if len(owner_repo) == 2:
            variants.append(Path(root) / owner_repo[0] / owner_repo[1] / rel_norm)
        variants.append(Path(root) / repo_key / rel_norm)
        for p in variants:
            if p.exists() and p.is_file():
                cand.append(p)
    return cand

def collect_yaml_index(yaml_dir: Path) -> Dict[str, List[Path]]:
    """Map: repo_key -> [yaml_file_paths] by filename prefix before '__'"""
    index: Dict[str, List[Path]] = {}
    for p in yaml_dir.iterdir():
        if p.is_file() and p.suffix.lower() in (".yml", ".yaml"):
            name = p.name.lower()
            if "__" in name:
                repo_key = name.split("__", 1)[0]
                index.setdefault(repo_key, []).append(p)
    return index

def main():
    # Load main CSV
    df = pd.read_csv(MAIN_CSV)
    df = lower_cols(df)

    # Normalize instru_t_ci_signal to boolean
    sig_col = "instru_t_ci_signal"
    if sig_col not in df.columns:
        raise KeyError(f"Required column '{sig_col}' not found in {MAIN_CSV}")

    def to_bool(x):
        if isinstance(x, str):
            return x.strip().lower() in ("true", "yes", "1")
        return bool(x)

    df["__signal"] = df[sig_col].apply(to_bool)
    df_sig = df[df["__signal"] == True].copy()

    yaml_dir = Path(YAML_DIR)
    if not yaml_dir.exists():
        raise FileNotFoundError(f"YAML directory not found: {YAML_DIR}")

    yaml_index = collect_yaml_index(yaml_dir)

    rows = []
    for _, rec in df_sig.iterrows():
        full_name = str(rec.get("full_name", "")).strip()
        if not full_name:
            continue
        repo_key = repo_key_from_full_name(full_name)

        # Accumulators per repo
        found_any_third_party = False
        acc_vals: Dict[str, List[str]]   = {k: [] for k in PARAM_PATTERNS.keys()}
        acc_tokens: Dict[str, List[str]] = {k: [] for k in PARAM_PATTERNS.keys()}
        provenance: Set[str] = set()

        for yml in yaml_index.get(repo_key, []):
            ytxt = read_text(yml)
            if not ytxt:
                continue

            # Third-party signals in YAML?
            if detect_third_party(ytxt):
                found_any_third_party = True
                vals, toks = extract_params_from_text_with_tokens(ytxt, PARAM_PATTERNS)
                merge_values(acc_vals, vals)
                merge_values(acc_tokens, toks)
                provenance.add(str(yml))

            # Follow referenced files from YAML (scripts/configs)
            for ref in find_file_refs_from_yaml(ytxt):
                # Locate in the repo clones
                refs = []
                for base in REPO_SEARCH_ROOTS:
                    refs.extend(find_repo_files(repo_key, [base], ref))

                for rp in refs:
                    rtxt = read_text(rp)
                    if not rtxt:
                        continue

                    if rp.suffix.lower() == ".json":
                        try:
                            j = json.loads(rtxt)
                            jtxt = json.dumps(j)
                            if detect_third_party(jtxt):
                                found_any_third_party = True
                            vals, toks = extract_params_from_text_with_tokens(jtxt, PARAM_PATTERNS)
                            merge_values(acc_vals, vals)
                            merge_values(acc_tokens, toks)
                            provenance.add(str(rp))
                            continue
                        except Exception:
                            pass

                    if rp.suffix.lower() in (".yml", ".yaml"):
                        try:
                            y = yaml.safe_load(rtxt)
                            ytxt2 = json.dumps(y, default=str)
                            if detect_third_party(ytxt2):
                                found_any_third_party = True
                            vals, toks = extract_params_from_text_with_tokens(ytxt2, PARAM_PATTERNS)
                            merge_values(acc_vals, vals)
                            merge_values(acc_tokens, toks)
                            provenance.add(str(rp))
                            continue
                        except Exception:
                            pass

                    if rp.suffix.lower() in (".sh", ".bash", ".bat", ".cmd", ".ps1"):
                        if detect_third_party(rtxt):
                            found_any_third_party = True
                        vals, toks = extract_params_from_text_with_tokens(rtxt, PARAM_PATTERNS)
                        merge_values(acc_vals, vals)
                        merge_values(acc_tokens, toks)
                        provenance.add(str(rp))

        # Only include repos where third-party lab usage was detected anywhere we looked
        if found_any_third_party:
            # finalize each parameter value + status
            target_device, target_status     = finalize_value_and_status(acc_vals["target_device"],   acc_tokens["target_device"])
            os_version_api, os_status        = finalize_value_and_status(acc_vals["os_version_api"],  acc_tokens["os_version_api"])
            app_apk_path, app_status         = finalize_value_and_status(acc_vals["app_apk_path"],    acc_tokens["app_apk_path"])
            test_apk_path, test_status       = finalize_value_and_status(acc_vals["test_apk_path"],   acc_tokens["test_apk_path"])
            auth_credentials, auth_status    = finalize_value_and_status(acc_vals["auth_credentials"], acc_tokens["auth_credentials"])

            rows.append({
                "full_name": full_name,
                "target_device": target_device,               "target_device_status": target_status,
                "os_version_api": os_version_api,             "os_version_api_status": os_status,
                "app_apk_path": app_apk_path,                 "app_apk_path_status": app_status,
                "test_apk_path": test_apk_path,               "test_apk_path_status": test_status,
                "auth_credentials": auth_credentials,         "auth_credentials_status": auth_status,
                "sources": "; ".join(sorted(provenance)) if provenance else "YAML only",
            })

    out_df = pd.DataFrame(rows).sort_values("full_name").reset_index(drop=True)
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved: {OUTPUT_CSV} (rows={len(out_df)})")
    if not out_df.empty:
        print(out_df.head(15).to_string(index=False))

if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Min_Parameters\5.1_ThirdPartyLab_Params_By_Repo.csv (rows=25)
                           full_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                target_device target_device_st

In [ ]:
# Looking for the Min Req Parameter for GMD
# -*- coding: utf-8 -*-
import os
import re
import json
import yaml
import pandas as pd
from pathlib import Path
from typing import Dict, List, Set

# === INPUTS ===
MAIN_CSV   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_Total_Repo_Dataset.csv"
CONFIG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Min_Parameters\5.2_GMD_Params_By_Repo.csv"

# --- GMD presence detection ---
GMD_DETECT_PATTERNS = [
    r'\bmanageddevices?\b',                # managedDevices / managedDevice
    r'\bManagedVirtualDevice\b',           # DSL type name
    r'\bmanageddevice\w*androidtest\b',    # task mentions
]

# --- Parameter patterns (explicit literals) ---
DEVICE_EXPLICIT = [
    r'(?mi)^\s*device\s*=\s*"([^"\n]+)"',
    r"(?mi)^\s*device\s*=\s*'([^'\n]+)'",
    r'(?mi)device\.set\(\s*"([^"\n]+)"\s*\)',
    r"(?mi)device\.set\(\s*'([^'\n]+)'\s*\)",
]
API_EXPLICIT = [
    r'(?mi)^\s*apiLevel\s*=\s*([0-9]{2,3})\b',
]
SYSIMG_EXPLICIT = [
    r'(?mi)^\s*systemImageSource\s*=\s*"([^"\n]+)"',
    r"(?mi)^\s*systemImageSource\s*=\s*'([^'\n]+)'",
    r'(?mi)systemImageSource\.set\(\s*"([^"\n]+)"\s*\)',
    r"(?mi)systemImageSource\.set\(\s*'([^'\n]+)'\s*\)",
]

# --- Fallback: capture RHS of assignments (to classify as env_or_input if not explicit) ---
DEVICE_ANY_RHS = r'(?mi)^\s*device\s*=\s*([^\n#;]+)'
API_ANY_RHS    = r'(?mi)^\s*apiLevel\s*=\s*([^\n#;]+)'
SYSIMG_ANY_RHS = r'(?mi)^\s*systemImageSource\s*=\s*([^\n#;]+)'

# Variable/secret/property references (Gradle & CI styles)
VAR_TOKEN_RE = re.compile(
    r'(?:(?:\$|\$\{)\s*[A-Za-z_][A-Za-z0-9_]*\s*\}?|'           # $VAR or ${VAR}
    r'\${{\s*(?:secrets|env|vars|inputs)\.[^}]+}}|'              # ${{ secrets.KEY }}
    r'%\([A-Za-z_][A-Za-z0-9_]*\)s|'                             # %(VAR)s
    r'\b(project\.|System\.getenv\(|findProperty\(|providers\.|extra\.)|'  # Gradle sources
    r'\$\{[^}]+\})'                                              # string interp
)

def read_text(p: Path) -> str:
    try:
        return p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        try:
            return p.read_text(encoding="latin-1", errors="ignore")
        except Exception:
            return ""

def lower_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def is_build_gradle(p: Path) -> bool:
    n = p.name.lower()
    return n.endswith("build.gradle") or n.endswith("build.gradle.kts") or n.endswith(".gradle") or n.endswith(".gradle.kts")

def is_yaml(p: Path) -> bool:
    return p.suffix.lower() in (".yml", ".yaml")

def repo_key_from_filename(p: Path) -> str:
    name = p.name.lower()
    return name.split("__", 1)[0] if "__" in name else ""

def has_gmd(text: str) -> bool:
    return any(re.search(pat, text, flags=re.I) for pat in GMD_DETECT_PATTERNS)

def find_explicit(text: str, patterns: List[str]) -> List[str]:
    vals: List[str] = []
    for pat in patterns:
        for m in re.finditer(pat, text):
            v = (m.group(1) or "").strip()
            if v and v not in vals:
                vals.append(v)
    return vals

def find_rhs(text: str, pattern: str) -> List[str]:
    vals: List[str] = []
    for m in re.finditer(pattern, text):
        rhs = (m.group(1) or "").strip()
        rhs = re.split(r'\s+#|//', rhs)[0].strip()
        if rhs and rhs not in vals:
            vals.append(rhs)
    return vals

def classify_values(explicit_vals: List[str], rhs_vals: List[str], is_api: bool=False):
    """
    (value_str, status):
      - explicit_vals present -> 'explicit'
      - else if rhs_vals present:
          - if rhs looks like var/secret/property -> 'env_or_input'
          - if api and rhs is integer -> 'explicit'
          - else -> 'env_or_input' (symbolic)
      - else -> 'unspecified'
    """
    if explicit_vals:
        return "; ".join(explicit_vals), "explicit"

    if rhs_vals:
        explicit_out: List[str] = []
        token_out: List[str] = []
        for rhs in rhs_vals:
            rhs_clean = rhs.strip().strip('"').strip("'")
            # numeric literal for api
            if is_api and re.fullmatch(r'[0-9]{2,3}', rhs_clean):
                if rhs_clean not in explicit_out:
                    explicit_out.append(rhs_clean)
                continue
            # variable/secret/property?
            if VAR_TOKEN_RE.search(rhs) or re.fullmatch(r'[A-Za-z_][A-Za-z0-9_.]*', rhs_clean):
                if rhs not in token_out:
                    token_out.append(rhs.strip())
            else:
                if rhs_clean and rhs_clean not in explicit_out:
                    explicit_out.append(rhs_clean)
        if explicit_out:
            return "; ".join(explicit_out), "explicit"
        if token_out:
            return "; ".join(token_out), "env_or_input"

    return "", "unspecified"

def build_index(config_dir: Path) -> Dict[str, Dict[str, List[Path]]]:
    """repo_key -> {'gradle': [...], 'yaml': [...]} based on filename prefix before '__'"""
    idx: Dict[str, Dict[str, List[Path]]] = {}
    for p in config_dir.iterdir():
        if not p.is_file():
            continue
        repo_key = repo_key_from_filename(p)
        if not repo_key:
            continue
        bucket = None
        if is_build_gradle(p):
            bucket = "gradle"
        elif is_yaml(p):
            bucket = "yaml"
        else:
            continue
        idx.setdefault(repo_key, {"gradle": [], "yaml": []})
        idx[repo_key][bucket].append(p)
    return idx

def main():
    # Load repos and restrict to instru_t_ci_signal = true/yes/1
    df = pd.read_csv(MAIN_CSV)
    df = lower_cols(df)

    if "full_name" not in df.columns or "instru_t_ci_signal" not in df.columns:
        raise KeyError("CSV must contain 'full_name' and 'instru_t_ci_signal'.")

    def to_bool(x):
        if isinstance(x, str):
            return x.strip().lower() in ("true", "yes", "1")
        return bool(x)

    df["__signal"] = df["instru_t_ci_signal"].apply(to_bool)
    df_sig = df[df["__signal"] == True].copy()

    config_dir = Path(CONFIG_DIR)
    if not config_dir.exists():
        raise FileNotFoundError(f"Config dir not found: {CONFIG_DIR}")

    index = build_index(config_dir)

    rows = []
    for _, rec in df_sig.iterrows():
        full_name = str(rec["full_name"]).strip()
        if not full_name:
            continue
        repo_key = full_name.lower().replace("/", ".")
        if repo_key not in index:
            continue

        gmd_present = False
        provenance: Set[str] = set()

        device_exp, api_exp, sysimg_exp = [], [], []
        device_rhs, api_rhs, sysimg_rhs = [], [], []

        # 1) Scan Gradle files — source of truth for GMD parameters
        for gp in index[repo_key]["gradle"]:
            gtxt = read_text(gp)
            if not gtxt:
                continue
            if has_gmd(gtxt):
                gmd_present = True
                provenance.add(str(gp))
                device_exp += [v for v in find_explicit(gtxt, DEVICE_EXPLICIT) if v not in device_exp]
                api_exp    += [v for v in find_explicit(gtxt, API_EXPLICIT)    if v not in api_exp]
                sysimg_exp += [v for v in find_explicit(gtxt, SYSIMG_EXPLICIT) if v not in sysimg_exp]

                device_rhs += [v for v in find_rhs(gtxt, DEVICE_ANY_RHS) if v not in device_rhs]
                api_rhs    += [v for v in find_rhs(gtxt, API_ANY_RHS)    if v not in api_rhs]
                sysimg_rhs += [v for v in find_rhs(gtxt, SYSIMG_ANY_RHS) if v not in sysimg_rhs]

        # 2) YAML only to acknowledge GMD mentions (values won’t be in YAML)
        for yp in index[repo_key]["yaml"]:
            ytxt = read_text(yp)
            if not ytxt:
                continue
            if has_gmd(ytxt):
                gmd_present = True
                provenance.add(str(yp))

        if not gmd_present:
            continue  # output only repos with detected GMD config

        # Finalize values with statuses
        device_val, device_status = classify_values(device_exp, device_rhs, is_api=False)
        api_val, api_status       = classify_values(api_exp,    api_rhs,    is_api=True)
        sysimg_val, sysimg_status = classify_values(sysimg_exp, sysimg_rhs, is_api=False)

        rows.append({
            "full_name": full_name,
            "device": device_val,                   "device_status": device_status,
            "api_level": api_val,                   "api_level_status": api_status,
            "system_image_source": sysimg_val,      "system_image_source_status": sysimg_status,
            "sources": "; ".join(sorted(provenance)) if provenance else "",
        })

    out_df = pd.DataFrame(rows).sort_values("full_name").reset_index(drop=True)
    Path(Path(OUTPUT_CSV).parent).mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved: {OUTPUT_CSV} (rows={len(out_df)})")
    if not out_df.empty:
        print(out_df.head(20).to_string(index=False))

if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Min_Parameters\5.2_GMD_Params_By_Repo.csv (rows=41)
                        full_name           device device_status    api_level api_level_status                        system_image_source system_image_source_status                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         sources
               andbible.and-bible          Pixel 3      explicit           31         explicit                                       ao

In [12]:
# Minimum Required Parameters for Real Device
# -*- coding: utf-8 -*-
import os
import re
import json
import yaml
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path
from typing import Dict, List, Set

# === INPUTS ===
MAIN_CSV   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_Total_Repo_Dataset.csv"
CONFIG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Min_Parameters\5.3_RealDevice_Params_By_Repo.csv"

# --- File type helpers ---
YAML_EXTS   = (".yml", ".yaml")
GRADLE_EXTS = ("build.gradle", "build.gradle.kts", ".gradle", ".gradle.kts")
SCRIPT_EXTS = (".sh", ".bash", ".bat", ".cmd", ".ps1")
XML_EXTS    = (".xml",)

def is_yaml(p: Path) -> bool:
    return p.suffix.lower() in YAML_EXTS

def is_gradle(p: Path) -> bool:
    n = p.name.lower()
    return n.endswith("build.gradle") or n.endswith("build.gradle.kts") or n.endswith(".gradle") or n.endswith(".gradle.kts")

def is_script(p: Path) -> bool:
    return p.suffix.lower() in SCRIPT_EXTS

def is_manifest(p: Path) -> bool:
    return p.suffix.lower() in XML_EXTS and "androidmanifest" in p.name.lower()

# --- Repo key mapping: filename prefix before '__' ---
def repo_key_from_filename(p: Path) -> str:
    name = p.name.lower()
    return name.split("__", 1)[0] if "__" in name else ""

# --- IO helpers ---
def read_text(p: Path) -> str:
    try:
        return p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        try:
            return p.read_text(encoding="latin-1", errors="ignore")
        except Exception:
            return ""

def lower_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [c.strip().lower() for c in df.columns]
    return df

# --- Variable/secret token detector ---
VAR_TOKEN_RE = re.compile(
    r'(?:(?:\$|\$\{)\s*[A-Za-z_][A-Za-z0-9_]*\s*\}?|'           # $VAR or ${VAR}
    r'\${{\s*(?:secrets|env|vars|inputs)\.[^}]+}}|'              # ${{ secrets.KEY }}
    r'%\([A-Za-z_][A-Za-z0-9_]*\)s)'                             # %(VAR)s
)

def is_token(val: str) -> bool:
    return bool(VAR_TOKEN_RE.search(val or ""))

def finalize_value_and_status(values_list: List[str], tokens_list: List[str]):
    """Return (value_str, status) according to what we actually found."""
    if values_list:
        return "; ".join(values_list), "explicit"
    if tokens_list:
        return "; ".join(tokens_list), "env_or_input"
    return "", "unspecified"

# --- Real-device detection (strong signals) ---
REAL_DEVICE_STRONG = [
    r'(?mi)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b',  # physical -s
    r'(?mi)^\s*adb\s+get-serialno\b',
    r'(?mi)^\s*adb\s+get-state\b',
]
# Emulator exclusions (to avoid miscounting emulator-only repos)
EMULATOR_STRONG = [
    r'(?mi)^\s*adb\s+-s\s+emulator-\d+\b',
    r'(?mi)^\s*adb\s+-s\s+(?:localhost|127\.0\.0\.1):\d+\b',
    r'(?mi)\breactivecircus/android-emulator-runner\b',
    r'(?mi)^\s*\S*emulator\b[^\n]*\s(-avd|@)\S+',
]

def has_real_device_signal(texts: List[str]) -> bool:
    blob = "\n".join(texts)
    if any(re.search(p, blob) for p in REAL_DEVICE_STRONG):
        return True
    # If only emulator signals are present, don't count as real-device
    if any(re.search(p, blob) for p in EMULATOR_STRONG):
        return False
    return False

# --- Parameter extractors ---

def extract_target_devices(text: str):
    """Return explicit serials/IPs and tokens for target device identifier."""
    explicit, tokens = [], []
    # adb -s <serial>
    for m in re.finditer(r'(?mi)^\s*adb\s+-s\s+(\S+)', text):
        dev = m.group(1).strip()
        # skip emulator forms
        if re.match(r'(?:emulator-\d+|localhost:\d+|127\.0\.0\.1:\d+)\b', dev, flags=re.I):
            continue
        (tokens if is_token(dev) else explicit).append(dev)
    # adb connect <ip:port>
    for m in re.finditer(r'(?mi)^\s*adb\s+connect\s+(\S+)', text):
        host = m.group(1).strip()
        (tokens if is_token(host) else explicit).append(f"connect:{host}")
    # dedup preserve order
    exp_out, tok_out = [], []
    for v in explicit:
        if v not in exp_out: exp_out.append(v)
    for v in tokens:
        if v not in tok_out: tok_out.append(v)
    return exp_out, tok_out

def extract_adb_connectivity(text: str):
    """Return list of connectivity actions found (values) and tokens (for connect host)."""
    values, tokens = [], []
    if re.search(r'(?mi)^\s*adb\s+devices\b', text):
        values.append("devices")
    if re.search(r'(?mi)^\s*adb\s+wait-for-device\b', text):
        values.append("wait-for-device")
    for m in re.finditer(r'(?mi)^\s*adb\s+connect\s+(\S+)', text):
        host = m.group(1).strip()
        if is_token(host):
            if f"connect:{host}" not in tokens:
                tokens.append(f"connect:{host}")
        else:
            if f"connect:{host}" not in values:
                values.append(f"connect:{host}")
    return list(dict.fromkeys(values)), list(dict.fromkeys(tokens))

def extract_apk_paths(text: str):
    """Return (app_apk_exp, app_apk_tok, test_apk_exp, test_apk_tok)."""
    app_exp, app_tok, test_exp, test_tok = [], [], [], []
    # From adb install lines
    for m in re.finditer(r'(?mi)^\s*adb\s+install(?:\s+-r)?\s+([^\s"\']+\.(?:apk|aab))', text):
        p = m.group(1).strip()
        is_test = bool(re.search(r'androidtest', p, flags=re.I))
        (test_tok if is_token(p) else test_exp if is_test else app_tok if is_token(p) else app_exp).append(p)
    # app:/test: style (YAML)
    for m in re.finditer(r'(?mi)\bapp(?:lication)?\s*[:=]\s*([^\s"\']+\.(?:apk|aab))', text):
        p = m.group(1).strip()
        (app_tok if is_token(p) else app_exp).append(p)
    for m in re.finditer(r'(?mi)\btest\s*[:=]\s*([^\s"\']+\.(?:apk|zip))', text):
        p = m.group(1).strip()
        (test_tok if is_token(p) else test_exp).append(p)
    # generic *.apk tokens
    for m in re.finditer(r'(?i)([^\s"\'=]+androidtest[^\s"\']*\.apk)', text):
        p = m.group(1).strip()
        (test_tok if is_token(p) else test_exp).append(p)
    for m in re.finditer(r'(?i)([^\s"\'=]+\.aab|[^\s"\'=]+\.apk)', text):
        p = m.group(1).strip()
        if re.search(r'androidtest', p, flags=re.I):
            continue  # already handled
        (app_tok if is_token(p) else app_exp).append(p)
    # de-dup preserve order
    def dedup(xs): 
        out=[]; [out.append(x) for x in xs if x not in out]; 
        return out
    return dedup(app_exp), dedup(app_tok), dedup(test_exp), dedup(test_tok)

def extract_test_command(text: str):
    """Return (commands_exp, commands_tok)."""
    exp, tok = [], []
    # ADB am instrument
    if re.search(r'(?mi)^\s*adb[^\n]*\bam\s+instrument\b', text):
        exp.append("adb am instrument")
    # Gradle connectedAndroidTest/deviceCheck
    for m in re.finditer(r'(?mi)^\s*(?:\.\/|\.\\)?gradle(?:w)?(?:\.bat)?[^\n]*\b(connected[a-z0-9:_-]*android[a-z0-9:_-]*test|connectedcheck|devicecheck|alldevicechecks)\b', text):
        task = m.group(1)
        exp.append(f"gradle: {task}")
    # Variable tokens near potential invocation lines
    for line in text.splitlines():
        if "gradle" in line.lower() or "adb" in line.lower():
            if VAR_TOKEN_RE.search(line):
                tok_line = VAR_TOKEN_RE.search(line).group(0)
                if tok_line not in tok:
                    tok.append(tok_line)
    return list(dict.fromkeys(exp)), list(dict.fromkeys(tok))

def extract_test_runner(texts_by_type: Dict[str, List[str]]):
    """
    Combine sources to determine test runner:
      1) from ADB 'am instrument' component
      2) Gradle: testInstrumentationRunner in build.gradle(.kts)
      3) AndroidManifest <instrumentation android:name="...">
    Return (values_exp, tokens)
    """
    exp, tok = [], []

    # 1) ADB 'am instrument' component: pkg/RunnerClass
    blob_all = "\n".join(sum(texts_by_type.values(), []))
    for m in re.finditer(r'(?mi)\bam\s+instrument[^\n]*\s([A-Za-z0-9_.]+/[A-Za-z0-9_.$]+)', blob_all):
        comp = m.group(1).strip()
        if comp not in exp:
            exp.append(comp)

    # 2) Gradle defaultConfig/testInstrumentationRunner
    gradle_blob = "\n".join(texts_by_type.get("gradle", []))
    for pat in [
        r'(?mi)^\s*testInstrumentationRunner\s*=\s*"([^"\n]+)"',
        r"(?mi)^\s*testInstrumentationRunner\s*=\s*'([^'\n]+)'",
        r'(?mi)^\s*testInstrumentationRunner\s+"([^"\n]+)"',
        r"(?mi)^\s*testInstrumentationRunner\s+'([^'\n]+)'",
        r'(?mi)testInstrumentationRunner\.set\(\s*"([^"\n]+)"\s*\)',
        r"(?mi)testInstrumentationRunner\.set\(\s*'([^'\n]+)'\s*\)",
    ]:
        for m in re.finditer(pat, gradle_blob):
            runner = m.group(1).strip()
            if is_token(runner):
                if runner not in tok: tok.append(runner)
            elif runner not in exp: exp.append(runner)
    # tokens near runner assignment
    for m in re.finditer(r'(?mi)^\s*testInstrumentationRunner[^\n]+', gradle_blob):
        line = m.group(0)
        if VAR_TOKEN_RE.search(line):
            t = VAR_TOKEN_RE.search(line).group(0)
            if t not in tok: tok.append(t)

    # 3) AndroidManifest <instrumentation android:name="...">
    for xmltxt in texts_by_type.get("manifest", []):
        try:
            root = ET.fromstring(xmltxt)
        except Exception:
            # fallback regex if XML parsing fails
            for m in re.finditer(r'(?i)<instrumentation[^>]*android:name\s*=\s*"([^"]+)"', xmltxt):
                nm = m.group(1).strip()
                if nm not in exp: exp.append(nm)
            continue
        ns = {"android": "http://schemas.android.com/apk/res/android"}
        for instr in root.iter():
            tag = instr.tag.split('}')[-1].lower()
            if tag == "instrumentation":
                name = instr.get("{http://schemas.android.com/apk/res/android}name")
                if name:
                    name = name.strip()
                    if name not in exp:
                        exp.append(name)

    return exp, tok

# --- Build repo index from CONFIG_DIR ---
def build_index(config_dir: Path) -> Dict[str, Dict[str, List[Path]]]:
    """
    repo_key -> {'yaml': [...], 'gradle': [...], 'scripts': [...], 'manifest': [...], 'other': [...]}
    Only files with names starting with 'owner.repo__' are indexed.
    """
    idx: Dict[str, Dict[str, List[Path]]] = {}
    for p in config_dir.iterdir():
        if not p.is_file():
            continue
        repo_key = repo_key_from_filename(p)
        if not repo_key:
            continue
        bucket = "other"
        if is_yaml(p):
            bucket = "yaml"
        elif is_gradle(p):
            bucket = "gradle"
        elif is_script(p):
            bucket = "scripts"
        elif is_manifest(p):
            bucket = "manifest"
        idx.setdefault(repo_key, {"yaml": [], "gradle": [], "scripts": [], "manifest": [], "other": []})
        idx[repo_key][bucket].append(p)
    return idx

def main():
    # Load repos, restrict to instru_t_ci_signal
    df = pd.read_csv(MAIN_CSV)
    df = lower_cols(df)
    if "full_name" not in df.columns or "instru_t_ci_signal" not in df.columns:
        raise KeyError("CSV must contain 'full_name' and 'instru_t_ci_signal'.")

    def to_bool(x):
        if isinstance(x, str):
            return x.strip().lower() in ("true", "yes", "1")
        return bool(x)

    df["__signal"] = df["instru_t_ci_signal"].apply(to_bool)
    df_sig = df[df["__signal"] == True].copy()

    config_dir = Path(CONFIG_DIR)
    if not config_dir.exists():
        raise FileNotFoundError(f"Config dir not found: {CONFIG_DIR}")

    index = build_index(config_dir)

    rows = []
    for _, rec in df_sig.iterrows():
        full_name = str(rec["full_name"]).strip()
        if not full_name:
            continue
        repo_key = full_name.lower().replace("/", ".")
        buckets = index.get(repo_key)
        if not buckets:
            continue

        # Collect texts by type for this repo
        texts_by_type: Dict[str, List[str]] = {k: [] for k in ["yaml", "gradle", "scripts", "manifest"]}
        sources: Set[str] = set()
        for typ in texts_by_type.keys():
            for p in buckets.get(typ, []):
                txt = read_text(p)
                if not txt:
                    continue
                texts_by_type[typ].append(txt)
                sources.add(str(p))

        if not any(texts_by_type.values()):
            continue

        # Decide if this repo has REAL DEVICE signals
        if not has_real_device_signal(texts_by_type["yaml"] + texts_by_type["scripts"]):
            # As a fallback, if 'am instrument' is present anywhere AND no strong emulator signals, assume real device usage.
            blob_all = "\n".join(sum(texts_by_type.values(), []))
            if not re.search(r'(?mi)\bam\s+instrument\b', blob_all):
                continue
            if any(re.search(p, blob_all) for p in EMULATOR_STRONG) and not any(re.search(p, blob_all) for p in REAL_DEVICE_STRONG):
                continue  # emulator-only

        # Extract parameters
        blob_yaml_scripts = "\n".join(texts_by_type["yaml"] + texts_by_type["scripts"])
        target_exp, target_tok = extract_target_devices(blob_yaml_scripts)
        conn_exp, conn_tok = extract_adb_connectivity(blob_yaml_scripts)

        app_exp, app_tok, test_exp, test_tok = extract_apk_paths(blob_yaml_scripts)

        cmd_exp, cmd_tok = extract_test_command(blob_yaml_scripts)

        runner_exp, runner_tok = extract_test_runner(texts_by_type)

        # Finalize value + status per parameter
        target_val, target_status = finalize_value_and_status(target_exp, target_tok)
        conn_val, conn_status     = finalize_value_and_status(conn_exp, conn_tok)
        app_val, app_status       = finalize_value_and_status(app_exp, app_tok)
        test_val, test_status     = finalize_value_and_status(test_exp, test_tok)
        cmd_val, cmd_status       = finalize_value_and_status(cmd_exp, cmd_tok)
        runner_val, runner_status = finalize_value_and_status(runner_exp, runner_tok)

        rows.append({
            "full_name": full_name,
            "target_device_identifier": target_val,     "target_device_identifier_status": target_status,
            "adb_connectivity": conn_val,              "adb_connectivity_status": conn_status,
            "app_apk_path": app_val,                   "app_apk_path_status": app_status,
            "test_apk_path": test_val,                 "test_apk_path_status": test_status,
            "test_command": cmd_val,                   "test_command_status": cmd_status,
            "test_runner": runner_val,                 "test_runner_status": runner_status,
            "sources": "; ".join(sorted(sources)),
        })

    out_df = pd.DataFrame(rows).sort_values("full_name").reset_index(drop=True)
    Path(Path(OUTPUT_CSV).parent).mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved: {OUTPUT_CSV} (rows={len(out_df)})")
    if not out_df.empty:
        print(out_df.head(20).to_string(index=False))

if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Min_Parameters\5.3_RealDevice_Params_By_Repo.csv (rows=8)
                 full_name target_device_identifier target_device_identifier_status adb_connectivity adb_connectivity_status                                                                                                                                                                                                                                                                                                      app_apk_path app_apk_path_status                                                                   test_apk_path test_apk_path_status                                                                                            test_command test_command_status                                                                                                                             test_runner test_runner_status                                        